In [ ]:
import os
import pandas as pd
import pickle
from string import punctuation

from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.classify import accuracy, NaiveBayesClassifier
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist

MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

In [ ]:
eng_stopwords = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def getTag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'

def lemmatizing(tokens):
    lemmatized = []
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, getTag(tag))
        lemmatized.append(result)
    return lemmatized

def preprocessing(sentence):
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in eng_stopwords]
    tokens = [token for token in tokens if token not in punctuation]
    tokens = [token for token in tokens if token.isalpha()]
    tokens = lemmatizing(tokens)
    return tokens

df = pd.read_csv(DATA_PATH)
x = df['Statement']
y = df['Sentiment']
allStatement = ' '.join(x)
allToken = preprocessing(allStatement)
freqDist = FreqDist(allToken)

def extract_feature(sentence):
    feature = {}
    for word in freqDist.keys():
        feature[word] = (word in sentence)
    return feature

feature_set = [(extract_feature(preprocessing(sentence)), sentiment)
               for sentence, sentiment in zip(x, y)
]

split = int(len(feature_set) * 0.8)
train_data = feature_set[:split]
test_data = feature_set[split:]


In [ ]:
def trainModel():
    model = NaiveBayesClassifier.train(train_data)
    model.show_most_informative_features(7)
    acc = accuracy(model, test_data)
    print(f"Accuracy : {acc*100}%")

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
    return model

In [1]:
def show_pos_tag(sentence):
    tokens = preprocessing(sentence)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word}: {tag}")
    return tokens

def show_synonyms_antonyms(tokens):
    for token in tokens:
        synonyms = []
        antonyms = []
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonyms.append(lemma.name())
                for ant in lemma.antonym():
                    antonyms.append(ant.name())
        if synonyms:
            print(f"Syn: {synonyms}")
        else:
            print("No Synonyms")
        if antonyms:
            print(f"Ant: {antonyms}")
        else:
            print("No antonyms")

def predict_statement(model, tokens):
    feature = extract_feature(tokens)
    prediction = model.classify(feature)
    print(f"Prediction: {prediction}")

def analyzeStatement(model, sentence):
    tokens = show_pos_tag(sentence)
    show_synonyms_antonyms(tokens)
    predict_statement(model, tokens)

In [2]:
sentence = ''

while True:
    print("1. Write")
    print("2. Analyze")
    print("3. Exit")
    choice = input("Choose your option: ")

    if choice == '1':
        sentence = input("Write your statement: ")
        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, 'rb') as f:
                model = pickle.load(f)
        else:
            model = trainModel()
            
    elif choice == '2':
        analyzeStatement(model, sentence)
    elif choice == '3':
        print("Exiting program")
        break
    else:
        print("Invalid option")

1. Write
2. Analyze
3. Exit


NameError: name 'os' is not defined